# Collinearity with Related Drivers
## When two drivers say the same thing

**The business question.** A used-car dealer prices every trade-in by its age and its mileage. Which of the two really drives resale value, and can both go in one pricing model? This notebook shows what happens when two drivers move almost in step, and what a manager should do about it.

You will fit the models in Python, read them in plain English, and check them against JMP. **Python leads and JMP is the check.**

### How to use this notebook

- Run cells in order with **Shift+Enter**. The data file is `../data/03_GuidedPractice_Car_Resale.csv`.
- Prices are in pounds. Mileage is in miles. Age is in years.
- Copilot is on. Where a cell says *Ask Copilot*, write a short prompt, run what it gives you, and check against the JMP numbers shown.

---

## Part 1: Demo: age and mileage

### Step 1: Load and look

Fifty used cars. Look at the columns before you decide what is a driver.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

cars = pd.read_csv("../data/03_GuidedPractice_Car_Resale.csv")
print(cars.shape)
cars.head()

In [ ]:
def fit_ols(data, y, drivers):
    # Fit y on a list of drivers. The constant is the intercept.
    return sm.OLS(data[y], sm.add_constant(data[drivers])).fit()

def vifs(data, drivers):
    # Variance inflation factor for each driver: how much the others overlap with it
    X = sm.add_constant(data[drivers])
    return pd.Series([variance_inflation_factor(X.values, i + 1) for i in range(len(drivers))], index=drivers).round(1)

cars[["Age_Years", "Mileage_Miles", "Resale_Value_GBP"]].describe().round(0)

### Step 2: Each driver on its own

Fit resale value on age, then on mileage. Both are strong, and both slopes are negative, as you would expect.

In [ ]:
age_only = fit_ols(cars, "Resale_Value_GBP", ["Age_Years"])
miles_only = fit_ols(cars, "Resale_Value_GBP", ["Mileage_Miles"])

print(f"Age alone:     each extra year   {age_only.params['Age_Years']:9.1f} pounds   R-squared {age_only.rsquared:.3f}")
print(f"Mileage alone: each extra mile   {miles_only.params['Mileage_Miles']:9.3f} pounds   R-squared {miles_only.rsquared:.3f}")

### Step 3: Put both in one model

An assistant would build the driver list like this: everything except the outcome. Run it and read the list it prints.

In [ ]:
drivers = [c for c in cars.columns if c != "Resale_Value_GBP"]
print("Drivers going into the model:", drivers)

both = fit_ols(cars, "Resale_Value_GBP", drivers)
pd.DataFrame({"Slope": both.params, "p-value": both.pvalues}).round(4)

Look at the list of drivers. Is every column in it a driver? `Car_ID` is only a row counter: it tells you where a car sits in the file and nothing about the car. Fix the list so it holds only age and mileage, re-run, and compare with the JMP numbers below.

In [ ]:
# Your fix goes here

**JMP check.** In JMP, **Analyze > Fit Model** with Resale Value as Y and Age and Mileage as the model effects. You should see an age slope of about **-2,019** (p about 0.02), a mileage slope of about **+0.103** (p about 0.41) and an R-squared of about **0.864**, adjusted about **0.858**.

Read that again. Alone, mileage is a strong negative driver. Together with age, its slope is **positive** (a higher-mileage car looks worth a little more) and not significant. That is not a finding about cars. It is what happens when two drivers carry the same information.

### Step 4: See the overlap

In [ ]:
print("Correlation between age and mileage:", round(cars["Age_Years"].corr(cars["Mileage_Miles"]), 3))
print("Variance inflation factors:", vifs(cars, ["Age_Years", "Mileage_Miles"]).to_dict())
print("With Car_ID wrongly included:", vifs(cars, ["Car_ID", "Age_Years", "Mileage_Miles"]).to_dict())

plt.figure(figsize=(5, 3.6))
plt.scatter(cars["Age_Years"], cars["Mileage_Miles"], s=16)
plt.xlabel("Age (years)"); plt.ylabel("Mileage (miles)"); plt.title("Age and mileage move together")
plt.show()

A correlation of 0.996 means that knowing a car's age almost tells you its mileage. The variance inflation factor (VIF) says how much the uncertainty in a slope is inflated by the overlap. A value around 10 is a warning; here it is about 120. With Car_ID wrongly included the two figures rise to about 132 and 135. Car_ID itself scores about 1, because a row counter overlaps with nothing, but it should never have gone in.

### Step 5: Why the slopes are unstable

If the model cannot tell age from mileage, small changes in which cars you sample should move the slopes a lot. Test that: fit the same model on five random sets of 35 of the 50 cars.

In [ ]:
rows = []
for seed in range(5):
    sample = cars.sample(35, random_state=seed)
    m = fit_ols(sample, "Resale_Value_GBP", ["Age_Years", "Mileage_Miles"])
    rows.append({"Sample": seed + 1, "Age slope": m.params["Age_Years"], "Mileage slope": m.params["Mileage_Miles"]})
pd.DataFrame(rows).round(3)

The age slope swings from about -1,500 to -2,700 pounds a year and the mileage slope from about 0.03 to 0.19 pounds a mile, positive every time although mileage alone is negative. The model fits well (R-squared 0.86) but it cannot say which driver deserves the credit, and which cars you happen to sample changes the answer.

### Step 6: Price an unusual car

A pricing model is only as good as the cars it was built on. Try a car that does not follow the usual pattern: three years old with 40,000 miles. Most three-year-old cars in the file have far fewer.

In [ ]:
odd_car = pd.DataFrame({"const": [1.0], "Age_Years": [3], "Mileage_Miles": [40000]})

for name, cols in [("Age only", ["Age_Years"]), ("Age and mileage", ["Age_Years", "Mileage_Miles"])]:
    m = fit_ols(cars, "Resale_Value_GBP", cols)
    print(f"{name:<16} predicted resale value: {m.predict(odd_car[['const'] + cols]).iloc[0]:,.0f} pounds")

print("\nCars aged 3 in the file: mileage runs from", cars.loc[cars["Age_Years"] == 3, "Mileage_Miles"].min(), "to", cars.loc[cars["Age_Years"] == 3, "Mileage_Miles"].max())

The two models disagree by over £1,600 on this car, and the model with both drivers says the high mileage *adds* value. Neither is trustworthy here: the car sits outside the pattern the data covers. Overlapping drivers make a model fragile exactly where the data is thin.

### Step 7: What should a manager do?

Three honest options:

1. **Keep age, drop mileage.** Simplest to explain, and almost nothing is lost in fit.
2. **Keep mileage, drop age.** Equally valid; mileage is the measurement a customer can check on the dashboard.
3. **Build one wear measure.** For example miles per year, which is not tied to age.

In [ ]:
cars["miles_per_year"] = cars["Mileage_Miles"] / cars["Age_Years"]

options = {
    "Age only": ["Age_Years"],
    "Mileage only": ["Mileage_Miles"],
    "Age and mileage": ["Age_Years", "Mileage_Miles"],
    "Age and miles per year": ["Age_Years", "miles_per_year"],
}
pd.DataFrame({name: {"R-squared": fit_ols(cars, "Resale_Value_GBP", c).rsquared,
                     "Adjusted R-squared": fit_ols(cars, "Resale_Value_GBP", c).rsquared_adj,
                     "Typical miss (pounds)": np.sqrt(fit_ols(cars, "Resale_Value_GBP", c).mse_resid)}
              for name, c in options.items()}).T.round(3)

In [ ]:
# Does miles per year overlap with age?
print("Correlation of age with miles per year:", round(cars["Age_Years"].corr(cars["miles_per_year"]), 2))
print("VIFs:", vifs(cars, ["Age_Years", "miles_per_year"]).to_dict())
m = fit_ols(cars, "Resale_Value_GBP", ["Age_Years", "miles_per_year"])
print("Miles-per-year slope p-value:", round(m.pvalues["miles_per_year"], 3))

All four options fit almost equally well (typical miss about £1,100). Miles per year no longer overlaps badly with age, but its slope is not significant either: it adds little. So the data supports the simplest choice. Keep **age** for the pricing model, say why, and mention that mileage tells the same story.

### Try this

Point the same code at a different pair of overlapping drivers. In the cell below, change the two columns to `Age_Years` and `miles_per_year`, then to `Mileage_Miles` and `miles_per_year`, and watch the VIFs.

In [ ]:
pair = ["Age_Years", "Mileage_Miles"]   # change this pair, then re-run

print("Correlation:", round(cars[pair[0]].corr(cars[pair[1]]), 3))
print("VIFs:", vifs(cars, pair).to_dict())
m = fit_ols(cars, "Resale_Value_GBP", pair)
print(m.params.round(3).to_dict())

### Commentary cell

1. Alone, mileage is a strong negative driver. With age in the model it turns positive. Explain, in plain words, why this does not mean high mileage is good for resale value.
2. What did the `Car_ID` column do to the model, and how did you spot it?
3. Which option would you recommend to the dealer, and why?

*Your commentary here*

### Say it to a manager

Two sentences, in business language. Say what drives resale value, and what you would not claim.

*Your two sentences here*

### Stretch

A blank space for you to try an idea of your own with Copilot. For example: draw the scatter plot of resale value against age coloured by mileage band, or find which cars the age-only model prices worst.

In [ ]:
# Stretch